Import Library and Register it in Spark

In [0]:
# Databricks notebook source
from myrestdatasource import MyRestDataSource
spark.dataSource.register(MyRestDataSource)

### 🔹 Basic API Read Test – JSONPlaceholder (No Auth, No Pagination)

In this test, we use a **public dummy REST API** provided by [JSONPlaceholder](https://jsonplaceholder.typicode.com/), which simulates typical CRUD operations and is widely used for testing and prototyping. The endpoint `/posts` returns a static array of 100 blog post objects in JSON format.

We configure our custom Spark DataSource (`myrestdatasource`) to fetch data from this endpoint. Since this API requires no authentication, pagination, or nested path navigation, it represents a simple and clean scenario to verify:

- That the connector can perform a basic GET request.
- That the response can be parsed into rows.
- That schema inference and automatic flattening work properly.

The test runs the `printSchema()` method to inspect the inferred schema and displays the resulting DataFrame.

This is the most basic test and serves as a good starting point to validate the overall functionality of the custom connector.

In [0]:
df_1 = (spark.read
      .format("myrestdatasource")
      .option("base_url", "https://jsonplaceholder.typicode.com")
      .option("endpoint", "posts")
      .load())

df_1.printSchema()
display(df_1)

root
 |-- userId: string (nullable = true)
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- body: string (nullable = true)



userId,id,title,body
1,1,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,quia et suscipit suscipit recusandae consequuntur expedita et cum reprehenderit molestiae ut ut quas totam nostrum rerum est autem sunt rem eveniet architecto
1,2,qui est esse,est rerum tempore vitae sequi sint nihil reprehenderit dolor beatae ea dolores neque fugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis qui aperiam non debitis possimus qui neque nisi nulla
1,3,ea molestias quasi exercitationem repellat qui ipsa sit aut,et iusto sed quo iure voluptatem occaecati omnis eligendi aut ad voluptatem doloribus vel accusantium quis pariatur molestiae porro eius odio et labore et velit aut
1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci sit amet autem assumenda provident rerum culpa quis hic commodi nesciunt rem tenetur doloremque ipsam iure quis sunt voluptatem rerum illo velit
1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed alias aut fugiat sit autem sed est voluptatem omnis possimus esse voluptatibus quis est aut tenetur dolor neque
1,6,dolorem eum magni eos aperiam quia,ut aspernatur corporis harum nihil quis provident sequi mollitia nobis aliquid molestiae perspiciatis et ea nemo ab reprehenderit accusantium quas voluptate dolores velit et doloremque molestiae
1,7,magnam facilis autem,dolore placeat quibusdam ea quo vitae magni quis enim qui quis quo nemo aut saepe quidem repellat excepturi ut quia sunt ut sequi eos ea sed quas
1,8,dolorem dolore est ipsam,dignissimos aperiam dolorem qui eum facilis quibusdam animi sint suscipit qui sint possimus cum quaerat magni maiores excepturi ipsam ut commodi dolor voluptatum modi aut vitae
1,9,nesciunt iure omnis dolorem tempora et accusantium,consectetur animi nesciunt iure dolore enim quia ad veniam autem ut quam aut nobis et est aut quod aut provident voluptas autem voluptas
1,10,optio molestias id quia eum,quo et expedita modi cum officia vel magni doloribus qui repudiandae vero nisi sit quos veniam quod sed accusamus veritatis error


### 🔹 Pagination Test – ReqRes API (Simulated Users with Page Support)

In this test, we use the [ReqRes API](https://reqres.in/), a public REST API designed specifically for testing HTTP clients. The endpoint `/api/users` returns a paginated list of user objects, with metadata such as `total_pages` and `per_page` included in the response.

We configure our custom Spark DataSource (`myrestdatasource`) to:

- Enable pagination (`pagination = true`)
- Start from page 1 (`start_page = 1`)
- Limit the maximum number of pages to 2 (`max_pages = 2`)
- Extract only the array of user objects using the JSON path `"data"`

This test verifies the connector's ability to:

- Loop through multiple pages via the configured page parameter (`?page=N`)
- Accumulate rows across API responses
- Parse a nested path (`data`) correctly
- Infer the schema from the first item on the first page

The schema is printed to validate correct field extraction, and the full DataFrame is displayed for manual inspection.

In [0]:
df_2 = (spark.read
      .format("myrestdatasource")
      .option("base_url", "https://reqres.in/api")
      .option("endpoint", "users")
      .option("pagination", "true")
      .option("start_page", "1")
      .option("max_pages", "2")
      .option("json_path", "data")
      .load())

df_2.printSchema()
display(df_2)

root
 |-- id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- avatar: string (nullable = true)



id,email,first_name,last_name,avatar
1,george.bluth@reqres.in,George,Bluth,https://reqres.in/img/faces/1-image.jpg
2,janet.weaver@reqres.in,Janet,Weaver,https://reqres.in/img/faces/2-image.jpg
3,emma.wong@reqres.in,Emma,Wong,https://reqres.in/img/faces/3-image.jpg
4,eve.holt@reqres.in,Eve,Holt,https://reqres.in/img/faces/4-image.jpg
5,charles.morris@reqres.in,Charles,Morris,https://reqres.in/img/faces/5-image.jpg
6,tracey.ramos@reqres.in,Tracey,Ramos,https://reqres.in/img/faces/6-image.jpg
7,michael.lawson@reqres.in,Michael,Lawson,https://reqres.in/img/faces/7-image.jpg
8,lindsay.ferguson@reqres.in,Lindsay,Ferguson,https://reqres.in/img/faces/8-image.jpg
9,tobias.funke@reqres.in,Tobias,Funke,https://reqres.in/img/faces/9-image.jpg
10,byron.fields@reqres.in,Byron,Fields,https://reqres.in/img/faces/10-image.jpg


### 🔹 Nested & Complex JSON Test – Random User API (Type Inference + Flattening)

In this test, we use the [Random User API](https://randomuser.me/), which generates realistic but fake user data in nested JSON format. The endpoint `/api/?results=3` returns a JSON object containing an array of 3 user objects under the `results` key.

We configure our custom Spark DataSource (`myrestdatasource`) to:

- Use the `results` key as the root array (`json_path = "results"`)
- Enable automatic data type inference (`infer_types = true`)

This test is designed to validate multiple advanced features of the connector:

- Navigation through nested JSON structures using `json_path`
- Flattening of deeply nested fields (e.g. `location.street.name`, `login.username`)
- Conversion of lists and objects into JSON strings when necessary
- Accurate detection of data types like `String`, `Boolean`, `Long`, `Timestamp`, etc.

By printing the schema and displaying the DataFrame, we confirm that all nested fields have been correctly flattened and typed, making this test ideal to verify robustness on real-world, semi-structured JSON APIs.


In [0]:
df_3 = (spark.read
    .format("myrestdatasource")
    .option("base_url", "https://randomuser.me")
    .option("endpoint", "api/?results=3")
    .option("json_path", "results")
    .option("infer_types", "true")
    .load())

df_3.printSchema()
display(df_3)

root
 |-- gender: string (nullable = true)
 |-- name.title: string (nullable = true)
 |-- name.first: string (nullable = true)
 |-- name.last: string (nullable = true)
 |-- location.street.number: long (nullable = true)
 |-- location.street.name: string (nullable = true)
 |-- location.city: string (nullable = true)
 |-- location.state: string (nullable = true)
 |-- location.country: string (nullable = true)
 |-- location.postcode: string (nullable = true)
 |-- location.coordinates.latitude: string (nullable = true)
 |-- location.coordinates.longitude: string (nullable = true)
 |-- location.timezone.offset: string (nullable = true)
 |-- location.timezone.description: string (nullable = true)
 |-- email: string (nullable = true)
 |-- login.uuid: string (nullable = true)
 |-- login.username: string (nullable = true)
 |-- login.password: string (nullable = true)
 |-- login.salt: string (nullable = true)
 |-- login.md5: string (nullable = true)
 |-- login.sha1: string (nullable = true)
 |--

gender,name.title,name.first,name.last,location.street.number,location.street.name,location.city,location.state,location.country,location.postcode,location.coordinates.latitude,location.coordinates.longitude,location.timezone.offset,location.timezone.description,email,login.uuid,login.username,login.password,login.salt,login.md5,login.sha1,login.sha256,dob.date,dob.age,registered.date,registered.age,phone,cell,id.name,id.value,picture.large,picture.medium,picture.thumbnail,nat
female,Mrs,Olga,Rolón,2666,Circunvalación Bhután,El Mármol,Morelos,Mexico,28612,25.7174,15.3748,+10:00,"Eastern Australia, Guam, Vladivostok",olga.rolon@example.com,cc7e4648-3f41-45ed-b735-f3add61ea18a,greenzebra316,citroen,7qgu5Z0B,38f2e7c1f16177dcd3bd75b0c5b0a23c,627fa2137d7842fb2096f324777d830081438286,2200f1537593bba455df7245d84230f0e6b3fa271ec6493f8095f535a66c9bd4,1981-09-30T23:20:51.122Z,43,2019-07-07T23:44:11.257Z,5,(631) 960 5688,(633) 711 3361,NSS,64 60 23 3934 9,https://randomuser.me/api/portraits/women/83.jpg,https://randomuser.me/api/portraits/med/women/83.jpg,https://randomuser.me/api/portraits/thumb/women/83.jpg,MX
male,Mr,پرهام,نكو نظر,5942,خاوران,زاهدان,کردستان,Iran,74751,66.9532,140.1153,+2:00,"Kaliningrad, South Africa",prhm.nkwnzr@example.com,54b12559-2979-4136-9e67-f91fc8937cb0,redrabbit906,xavier,bkGAIrs0,ba1e191355a79a6708adc6a8e2c0031b,de5da2debedbfe53b81b8bc31044bb2c916f0091,c66e40ac9c138a6e2429a5baeb6d7a7ec0680888b803d6b3aedf9147723902a2,1996-06-13T06:10:00.177Z,28,2011-03-08T21:00:31.671Z,14,029-32766879,0924-900-7081,,null,https://randomuser.me/api/portraits/men/32.jpg,https://randomuser.me/api/portraits/med/men/32.jpg,https://randomuser.me/api/portraits/thumb/men/32.jpg,IR
female,Miss,Freja,Møller,7402,Plantagevej,Snertinge,Sjælland,Denmark,34277,-58.6355,9.4560,+6:00,"Almaty, Dhaka, Colombo",freja.moller@example.com,6844ffac-ee75-44e5-bc99-dbf47f00b1d2,angryfish327,lillie,9DJSk59F,2de3cd27697b166008f2479b5fd8b281,30dca3bba5c0ffcb9e98b6b8f128f87958e77119,efe0c6a051cfbf5cb54e185c6dd87b2788bb2e521444d3d1ec5071c68def28d2,1976-11-30T01:44:32.596Z,48,2007-07-23T15:17:44.986Z,17,60516652,07165236,CPR,291176-8462,https://randomuser.me/api/portraits/women/17.jpg,https://randomuser.me/api/portraits/med/women/17.jpg,https://randomuser.me/api/portraits/thumb/women/17.jpg,DK


### 🔹 Protected API Test – Valid Token using Postman Mock Server

This test verifies that the custom Spark data source can properly authenticate using a valid Bearer token and correctly parse a response from a mock API. The data source is configured to call a **GET** request on the Postman Mock Server endpoint, which is set up to return a valid response only when the Authorization header contains the token `Bearer test123`.

**Data Source Details:**
- **Base URL:** `https://xxx.mock.pstmn.io`
- **Endpoint:** `get`
- **Authentication:** Uses an Authorization header set to `Bearer test123`
- **Response Parsing:** The response from the mock server is expected to have a JSON structure with data nested under `args.data`; for example, a typical response could be:
  ```json
  {
    "args": {
      "data": [
        { "id": 1, "role": "admin" }
      ]
    },
    ...
  }


In [0]:
df_token_ok = (spark.read
    .format("myrestdatasource")
    .option("base_url", "https://xxx.mock.pstmn.io")
    .option("endpoint", "get")
    .option("auth_token", "Bearer test123")
    .option("json_path", "args.data")
    .option("infer_types", "true")
    .load())

df_token_ok.printSchema()
display(df_token_ok)

root
 |-- id: long (nullable = true)
 |-- role: string (nullable = true)



id,role
1,admin
